# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org//) et [PDF](https://python.guillod.org//python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?filepath=chap07.ipynb).

# 7 Zéro de fonctions

<div id="ch:newton"></div>

Le but de cette série d'exercices est de déterminer les zéros de manière approchée d'une fonction notamment par la méthode de Newton. Cela permet en particulier de trouver des solutions approchées d'équations non-linéaires. Cette méthode est fondamentale autant d'un point de vue numérique qu'analytique.

**Concepts abordés:**

* méthode de Newton en une et plusieurs dimensions

* matrice jacobienne

* attracteur de la méthode de Newton

* ensemble fractal

* optimisation par parallélisation

* équation différentielle non-linéaire

* différences finies



# Exercice 7.1: Méthode de Newton en une dimension

En une dimension, la méthode de Newton consiste à trouver une solution approchée d'une seule équation. Cette équation peut être mise sous la forme générale $F(x)=0$ où $F:\mathbb{R}\to\mathbb{R}$ est une fonction assez régulière, donc le but est de trouver un zéro de la fonction $F$. L'équation $F(x)=0$ est équivalente (si $F^\prime(x) \neq 0$) à l'équation $G(x)=x$ où $G$ est la fonction définie par:

$$
G(x) = x - \frac{F(x)}{F^\prime(x)} \,.
$$

La méthode de Newton consiste à trouver un point fixe de $G$, *i.e.* résoudre $G(x)=x$ par itérations successives:

$$
x_{i+1} = G(x_i) = x_i - \frac{F(x_i)}{F^\prime(x_i)}
$$

à partir d'une valeur initiale $x_0\in\mathbb{R}$.
Lorsque la suite $(x_i)_ {i\in\mathbb{N}}$ converge, alors la limite $x$ est une solution de $G(x)=x$ donc de $F(x)=0$.

**a)**
Écrire une fonction `newton1d(F, DF, x0, eps=1e-10, N=1000)` qui donné une fonction $F$, sa dérivée $F^\prime$ et une valeur initiale $x_0$ calcule les itérations de Newton jusqu'à ce que  $|F(x_{i})| < \varepsilon$ et retourne $x_{i}$. Si $N$ itérations n'ont pas suffi à atteindre ce critère de convergence, alors retourner une erreur.



In [2]:
import numpy as np
from sympy import *
init_printing()
x=Symbol("x",real=True)
y=Symbol("y",real=True)
j = Symbol('j',real = True)
k = Symbol('k',real = True)

In [6]:
def newton1d(F, DF, x0,  eps=1e-10, N=1000):
    error = 'ne converge pas'
    while N > 0 and abs(F.subs(x, x0)) >= eps:
        x1 = x0 - F.subs(x, x0) / DF.subs( x, x0) # hypothèse, DF.subs( x, x0) !=0
        x0 = x1
        N = N - 1
    if N > 0:
        print (f'nombre itérations: {1000-N}')
        return x0, None
    else:
        return x0, error

F = x**3+3.0*x**2-6*x-8
DF = diff(F, x)
x0 = 0
print(f'fonction: {F}, x0: {newton1d(F, DF, x0)}')

nombre itérations: 4
fonction: x**3 + 3.0*x**2 - 6*x - 8, x0: (-1.00000000000000, None)



**b)**
En utilisant la fonction définie précédemment, trouver une solution approchée de l'équation $e^{-x} = x$.

**Réponse:**
La solution est approximativement donnée par $x=0.56714$.



In [7]:
F = exp(-x)-x
DF = diff(F, x)
x0 = 0
r, error = newton1d(F, DF, x0)
if error == None:
    display(r.evalf())
else:
    display(error)

nombre itérations: 4


Ce petit programme, basé sur la fonction newtond1, qui est basée à son tour sur sympy, me semble très lent. Pour ce cas particulier, le code suivant parait plus approprié en question de rapidité.

In [ ]:
import math
F = lambda x: math.exp(-x)-x
DF = lambda x: -1-(math.exp(-x))
def newton1d1(F,DF,x0,eps=1e-10,N=10):
    xn = x0
    for _ in range(0,N+1):
        if abs(F(xn)) < eps:
            return xn
        xn = xn - F(xn)/DF(xn) # hypothèse, DF(xn)!=0
    return None

print(newton1d1(F,DF,1))

0.5671432904097838



**c)**
Sans utiliser la fonction `sqrt` ni les puissances fractionnaires, définir une fonction `racine(x,n)` qui calcule $\sqrt[n]{x}$.



In [ ]:
def racine(y,n):
    # j'appele le radicande 'y' pour ne pas avoir des conflits avec les symbol sympy 'x'
    # cela consiste a trouver un zéro de la fonction F = x**n - y
    F = x**n - y
    DF = diff(F, x)
    r, error = newton1d(F, DF, 1)
    if error == None:
        return r.evalf()
    else:
        print(error)
        return None

#test
r = racine(27,3)
if  r != None:
    print(f'racine: {r}')

nombre itérations: 8
racine: 3.00000000000010


Cependant, cette fonction racine, qui est basée sur newton1d est plutôt rapide comparée à l'exemple du point b


Parfois, la dérivée de la fonction $F$ n'étant pas calculable analytiquement, il est alors nécessaire de l'approcher numériquement:

$$
F^\prime(x_i) \approx \frac{F(x_i) - F(x_{i-1})}{x_i-x_{i-1}}
$$

ce qui mène à la méthode de la sécante:

$$
x_{i+1} =  x_i - F(x_i) \frac{x_i-x_{i-1}}{F(x_i) - F(x_{i-1})} = \frac{x_{i-1}F(x_{i})-x_{i}F(x_{i-1})}{F(x_{i})-F(x_{i-1})}
$$

où $x_0$ et $x_1$ doivent être choisi.

**d)**
Écrire une méthode `secant1d(F, x0, x1, eps=1e-10, N=1000)` implémentant la méthode de la sécante et la tester sur l'exemple précédent.



In [ ]:
def secant1d(F, x0, x1, eps=1e-10, N=1000):
    error = 'ne converge pas'
    while N > 0 and abs(F.subs(x, x0)) >= eps:
        # print(f'x0: {x0}\tx1: {x1}')
        x2 = (x0*F.subs(x, x1) - x1*F.subs(x, x0)) / (F.subs(x, x1) - F.subs(x, x0))
        x0 = x1
        x1 = x2
        N = N - 1
    print (f'nombre itérations: {1000-N}')
    if N > 0:   
        return x0, None
    else:
        return x0, error

F = x**3+3.0*x**2-6*x-8
x0 = 0
x1 = 5
print(f'fonction: {F}, x0: {secant1d(F, x0, x1)}')
F = x**3-27
x0 = 1.0
x1 = 5
print(f'fonction: {F}, x0: {secant1d(F, x0, x1)}')

nombre itérations: 8
fonction: x**3 + 3.0*x**2 - 6*x - 8, x0: (-1.00000000000000, None)
nombre itérations: 10
fonction: x**3 - 27, x0: (3.00000000000000, None)


# Exercice 7.2: Méthode de Newton en plusieurs dimensions

<div id="exer:newton-Rn"></div>

La méthode de Newton en une dimension est facilement généralisable à plusieurs dimensions pour résoudre des équations de la forme $F(\boldsymbol{x})=\boldsymbol{0}$ où $F:\mathbb{R}^n \to \mathbb{R}^n$ est une fonction assez régulière. Conceptuellement la méthode est identique: l'équation $F(\boldsymbol{x})=\boldsymbol{0}$ est équivalente à $G(\boldsymbol{x})=\boldsymbol{x}$ où:

$$
G(\boldsymbol{x}) = \boldsymbol{x} - \big(F^\prime(\boldsymbol{x})\big)^{-1} F(\boldsymbol{x})
$$

où $F^\prime(\boldsymbol{x})$ désigne la matrice jacobienne de taille $n \times n$ de $F$ en $x$.
Ainsi les itérations de Newton s'écrivent:

$$
\boldsymbol{x}_{i+1} = \boldsymbol{x}_{i} - \big(F^\prime(\boldsymbol{x}_i)\big)^{-1} F(\boldsymbol{x}_i)
$$

**a)**
Écrire une fonction `newton(F, DF, x0, eps=1e-12, N=10000)` implémentant la méthode de Newton en plus qu'une dimension.

**Indication:**
Faire attention que les listes et les tableaux sont mutables. Pour avoir une performance optimale, il ne faut pas inverser la matrice jacobienne mais résoudre un système linéaire avec $F(\boldsymbol{x}_i)$ comme second membre.



In [ ]:
def newton(F, DF, x0, eps=1e-12, N=10000):
    """ array(2,1) x array(2,2) x array(2,1) ---> array(2,1)
        Hypothèses: F contient l'expression de F1(x,y) et F2(x,y), DF contient les dérivées partielles en x et y de F et x0 vecteur initial
        Méthode de Newton pour deux dimensions, retourne le vecteur solution si trouvé, sinon None """
    
    Y = Matrix([x, y])
    J = Matrix([j, k])
    D = (Y - J)
    A = DF*D
    nextx = x0[0]
    nexty = x0[1]
    vector = np.zeros(2,dtype=float)
    for i in range(N):
        A1 = A.subs({x:nextx,y:nexty})
        b = F.subs({x:nextx,y:nexty})
        eqs = A1-b # forme les équations linéaires comme demandé
        sols = solve((eqs[0],eqs[1]),j,k)
        vector[0] = sols[j].evalf()
        vector[1] = sols[k].evalf()
        if abs(np.linalg.norm(np.array([vector[0],vector[1]])) - np.linalg.norm(np.array([nextx,nexty]))) < eps:
            break
        nextx = vector[0]
        nexty = vector[1]
    if i == N-1:
        print(f'ne converge pas')
        return None
    else:
        print(f'solution {vector} trouvée en {i+1} itérations')
        return vector

Avec une tolerance de 1e-12 cet example ne converge jamais, je suis obligé de mettre eps=1e-9 ou supérieur

In [ ]:
X = Matrix([x**2+y**2-4, 4*x**2-y**2-4])
V = Matrix([x, y])
Z = X.jacobian(V)
X0 = np.array([1,1])
newton(X,Z,X0,eps=1e-9)

solution [1.26491106 1.54919334] trouvée en 5 itérations


array([1.26491106, 1.54919334])

In [ ]:
X = Matrix([x**3-5*x**2+2*x-y+13, x**3+x**2-14*x-y-19])
V = Matrix([x, y])
Z = X.jacobian(V)
X0 = np.array([8,10])
newton(X,Z,X0)
X0 = np.array([-2,12])
newton(X,Z,X0)

solution [4. 5.] trouvée en 22 itérations
solution [-1.33333333 -0.92592593] trouvée en 5 itérations


array([-1.33333333, -0.92592593])


**b)**
Utiliser la fonction précédente pour résoudre le système suivant:

$$
\begin{align*}
\cos(x) & =\sin(y)\,, & e^{-x} & =\cos(y)\,.
\end{align*}
$$

In [ ]:
X = Matrix([cos(x)-sin(y), exp(-x)-cos(y)])
V = Matrix([x, y])
Z = X.jacobian(V)
X0 = np.array([1,1])
newton(X,Z,X0)

solution [0.58853274 0.98226358] trouvée en 5 itérations


array([0.58853274, 0.98226358])


**Réponse:**
Une solution est approximativement donnée par $x=0.58853$ et $y=0.98226$.



# Exercice 7.3: Attracteur de la méthode de Newton

Le but de cet exercice est de résoudre l'équation $z^3 = 1$ dans le plan complexe à l'aide de la méthode de Newton et d’analyser vers laquelle des trois racines de l'unité la méthode va converger suivant le choix du point initial $z_0$.

**a)**
Si nécessaire adapter la fonction `newton1d` pour que celle-ci marche également pour des nombres complexes et la tester pour résoudre $z^3 = 1$ à partir de différentes valeurs de $z_0$.




**b)**
Déterminer pour chaque $z_0 \in \big\{x_0+iy_0: x_0\in[-3,3] \; \text{et} \; y_0\in[-3,3]\big\}$ vers quelle racine de l'unité la méthode de Newton va converger. Représenter graphiquement cet ensemble.

**Indication:**
La fonction `meshgrid` de Numpy peut être utile pour construire la matrice correspondante à l'ensemble des $z_0$.

**Réponse:**
Le graphique donnant la convergence de $z_0$ vers une des trois racines est le suivant:

<center><img src="https://python.guillod.org/fig/newton-z3.png" style="width:90%;max-width:600px;"></center>




**c)**
<span style="color:red">!</span> La méthode précédente a le désavantage de procéder séquentiellement au calcul pour chaque valeur de $z_0$, ce qui rend cette évaluation assez lente. Proposer une nouvelle implémentation permettant de calculer parallèlement toutes les valeurs de $z_0$ en utilisant les indexages Numpy.

**Indication:**
Pour encore plus de rapidité, les itérations de Newton de $F(z)=z^3-1$ peuvent être calculées à la main:

$$
z_{n+1} = \frac{1}{3z_n^2} + \frac{2z_n}{3} \,.
$$



<!-- --- end exercise --- -->




<!-- --- begin exercise --- -->

# Exercice 7.4: <span style="color:red">!!</span> Équation différentielle non-linéaire

<div id="exer:newton-nonlinear"></div>

Le but est de résoudre l'équation différentielle suivante avec conditions de valeurs limites:

$$
\begin{align*}
u^{\prime\prime}(x)+u^{3}(x) & = \sin(x) \,, & u(0) & = u(2\pi) = 0 \,,
\end{align*}
$$

sur l'intervalle $[0,2\pi]$. Cette équation est un modèle simplifié pour une équation de Schrödinger non-linéaire.

La méthode employée est les différences finies qui consistent à chercher les valeurs de $u$ aux points $x_n = \frac{2\pi n}{N}$ pour $n=0,1,\dots,N$. Les inconnues sont alors les nombres $u_n = u(x_n)$ et forment un vecteur de dimension $N+1$. La méthode des différences finies consiste à approximer la dérivée seconde par

$$
u^{\prime\prime}(x) \approx \frac{u(x+h)-2u(x)+u(x-h)}{h^{2}} \,,
$$

lorsque $h$ est petit.
En prenant $h=\frac{2\pi}{N}$, alors:

$$
u^{\prime\prime}(x_n) \approx \frac{u_{n+1}-2u_n+u_{n-1}}{h^{2}} \,,
$$

et donc l'équation initiale s'approxime par:

$$
\begin{align*}
\frac{u_{n+1}-2u_{n}+u_{n-1}}{h^{2}}+u_{n}^{3} & = \sin(x_{n}) \,, & u_{0} & = u_{N} = 0 \,,
\end{align*}
$$

pour $n=1,2,\dots,N-1$.
Cette équation peut être vue comme une équation du type $F(\boldsymbol{u})=\boldsymbol{0}$ pour $\boldsymbol{u} = (u_n)_ {n=0}^{N+1}$ et donc être résolue par la méthode de Newton.

**a)**
Montrer l'approximation suivante:

$$
u^{\prime\prime}(x) = \frac{u(x+h)-2u(x)+u(x-h)}{h^{2}} + O(h^2) \quad \text{lorsque} \quad h \to 0 \,.
$$



**Indication:**
Utiliser le théorème de Taylor.





**b)**
Définir un vecteur `x` représentant les $N+1$ points équidistribués dans $[0,2\pi]$ et `h` la distance entre les points, avec par exemple $N=200$.




**c)**
Définir une fonction `F(u)` représentant la fonction $F:\mathbb{R}^{N+1} \to \mathbb{R}^{N+1}$ permettant de mettre l'équation approchée sous la forme $F(\boldsymbol{u})=\boldsymbol{0}$.

**Indication:**
Pour avoir une implémentation rapide, il est impératif de ne pas utiliser une boucle pour construire $F$ mais d'utiliser le slicing Numpy.




**d)**
Définir une fonction `DF(u)` représentant la jacobienne de la fonction précédente.

**Indication:**
La jacobienne est la dérivée de $F(\boldsymbol{u})=F(u_0,u_1,\dots,u_{N})$ par rapport à $\boldsymbol{u} = (u_0,u_1,\dots,u_{N})$, c'est à dire:

$$
F^{\prime}(\boldsymbol{u})=\begin{pmatrix}\partial_{0}F(\boldsymbol{u}) & \partial_{1}F(\boldsymbol{u}) & \partial_{2}F(\boldsymbol{u}) & \cdots & \partial_{N-1}F(\boldsymbol{u}) & \partial_{N}F(\boldsymbol{u})\end{pmatrix}\,,
$$

et peut se calculer explicitement à la main.





**e)**
Utiliser la fonction `newton` définie précédemment pour calculer une solution approchée de l'équation. En changeant de valeurs initiales, est-il possible de trouver d'autres solutions ?

**Indication:**
Essayer avec la donnée initiale $u_0(x) = (1+k)\sin(kx)$ pour $k=1,2,3,4$ comme point de départ de la méthode de Newton.

